## Path configuration

In [ ]:
from pathlib import Path
import os

PROJECT_NAME = "MALDIAlign"

cwd = Path().resolve()

# Walk upwards until we find the project folder
target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

# If the project folder is found and we are not already there, then change cwd
if target is not None and target != cwd:
    os.chdir(target)

print("Working directory:", os.getcwd())

Working directory: /export/usuarios01/agnavarr/MALDIAlign


## Imports

In [ ]:
import os
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.load_config import load_config
from utils.load_data import load_pkl

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## Data preparation

### Data loading

In [ ]:
cfg = load_config()
driams_pkl = cfg["data"]["DRIAMS_REDUCED_PKL"]

In [ ]:
driams = load_pkl(driams_pkl)

In [ ]:
data, label, meta = driams["data"], driams["label"], driams["meta"]

In [ ]:
meta = pd.DataFrame.from_records(list(meta))

### Filter data and construct final dataset

In [ ]:
maskA = np.where(meta["hospital"].values == "DRIAMS_A")[0]
dataA, labelA, metaA = data[maskA], label[maskA], meta.iloc[maskA]

maskD = np.where(meta["hospital"].values == "DRIAMS_D")[0]
dataD, labelD, metaD = data[maskD], label[maskD], meta.iloc[maskD]

In [ ]:
# Downsample DRIAMS-A
from sklearn.model_selection import train_test_split

_, dataA_sub, _, labelA_sub, _, metaA_sub = train_test_split(
    dataA,
    labelA,
    metaA,
    test_size=len(dataD),
    random_state=42,
    stratify=labelA)

In [ ]:
# Concatenate data and labels
data_final = np.vstack([dataA_sub, dataD])
label_final = np.concatenate([labelA_sub, labelD])
meta_final  = pd.concat([metaA_sub, metaD], ignore_index=True)

In [ ]:
species, counts = np.unique(labelA_sub, return_counts=True)
for sp, n in zip(species, counts):
    print(f"{sp}: {n}")

Enterobacter_cloacae_complex: 676
Enterococcus_Faecium: 473
Escherichia_Coli: 1940
Klebsiella_Pneumoniae: 1073
Pseudomonas_Aeruginosa: 1290
Staphylococcus_Aureus: 1856


In [ ]:
species, counts = np.unique(labelD, return_counts=True)
for sp, n in zip(species, counts):
    print(f"{sp}: {n}")

Enterobacter_cloacae_complex: 437
Enterococcus_Faecium: 171
Escherichia_Coli: 2013
Klebsiella_Pneumoniae: 2151
Pseudomonas_Aeruginosa: 362
Staphylococcus_Aureus: 2174


### Construct dataloaders

In [ ]:
X_train, X_val, y_train, y_val, meta_train, meta_val = train_test_split(
    data_final, label_final, meta_final,
    test_size=0.2, random_state=42, stratify=label_final
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val, dtype=torch.float32)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, torch.zeros(len(X_train)))
val_dataset   = TensorDataset(X_val_tensor, torch.zeros(len(X_val)))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

## VAE Multi-decoder

### Architecture

In [103]:
class CommmonEncoder(nn.Module):
  def __init__(self, input_dim, latent_dim):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.ReLU(),
        nn.Linear(128, 512),
        nn.ReLU(),
        nn.Linear(512, latent_dim)
    )

  def forward(self, x):
    hidden_rep = self.net(x)
    mu = self.mu(hidden_rep)
    logvar = self.logvar(hidden_rep)
    logvar = torch.clamp(logvar, -6, 6)
    return mu, logvar

In [110]:
class Decoders(nn.Module):
  def __init__(self, latent_dim, output_dim, num_domains):
    super().__init__()
    self.net = nn.ModuleList([
      nn.Sequential(
        nn.Linear(latent_dim, 256),
        nn.ReLU(),
        nn.Linear(256, 512),
        nn.ReLU(),
        nn.Linear(512, output_dim)
    ) for _ in range(num_domains)
    ])

  def forward(self, z, domain_id):
    return self.net[domain_id](z)

In [118]:
class MultiVAE(nn.Module):
  def __init__(self, input_dim, latent_dim, num_domains):
    super().__init__()
    self.encoder = CommonEncoder(input_dim, latent_dim)
    self.decoder = Decoders(latent_dim, input_dim, num_domains)

  def reparameterize(self, mu, logvar):
    std = torch.exp(0.5*logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

  def forward(self, x, domain_id):
    mu, logvar = self.encoder(x)
    z = self.reparameterize(mu, logvar)
    x_recon = self.decoder(z, domain_id)
    return x_recon, mu, logvar

In [ ]:
class MultiVAE_Extended(MultiVAE):
  def __init__(self, input_dim, latent_dim, num_domains, epochs=100, lr=1e-4, annealing_epochs=50):
    self.epochs = epochs
    self.lr = lr
    self.annealing_epochs = annealing_epochs

    self.optimizer = optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-5)
    self.criterion = nn.MSELoss()

    self.loss_during_training = []
    self.reconstruc_during_training = []
    self.KL_during_training = []

    def loss_function(self, x, x_recon, mu, logvar, beta):
      recon_loss = self.criterion(x_recon, x)
      kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
      total_loss = recon_loss + beta * kl_loss
      return total_loss, recon_loss, kl_loss

  def trainloop(self, trainloader, validloader, device):


### Training

### Plot results

### t-SNE